# FFAI History & Analytics: The Basics

This notebook demonstrates FFAI's four history stores and basic analytics:

1. **Run a multi-step pipeline** -- 5 named prompts that build on each other
2. **Raw history** -- the full interaction record including usage and status
3. **Ordered history** -- sequenced view with cleaned prompts and responses
4. **Prompt-attribute history** -- indexed by prompt name for `{{name.response}}` interpolation
5. **Basic statistics** -- usage counts, response lengths, token/cost aggregation

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="coroutine.*was never awaited", category=RuntimeWarning)

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / "pyproject.toml").is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

from ffai import FFAI  # noqa: E402
from ffai.Clients import FFLiteLLMClient  # noqa: E402

client = FFLiteLLMClient(model_string="mistral/mistral-small-latest")
ffai = FFAI(client=client)

print(f"Project root: {_project_root}")
print(f"Client model: {client.model}")
print("Ready")

11:10:55 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


11:10:56 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Project root: /home/aq/Documents/Source/ffai-standalone
Client model: mistral-small-latest
Ready


<div class="page-break"></div>

---

## Step 1: Run a 5-Prompt Pipeline

Generate a chain of named prompts. Each builds on the previous one via
`{{name.response}}` interpolation. This populates all four history stores.

In [2]:
r1 = ffai.generate_response(
    prompt="List exactly four software architecture patterns with a one-sentence description of each.",
    prompt_name="patterns",
)
print(f"[patterns] tokens={r1.usage.total_tokens} cost=${r1.cost_usd:.6f}")
print(r1.response[:200])
print("...")

[patterns] tokens=184 cost=$0.000028
1. **Layered Architecture**: Organizes a system into layers (e.g., presentation, business logic, data access) with dependencies flowing downward, promoting separation of concerns and modularity.

2. *
...


In [3]:
r2 = ffai.generate_response(
    prompt="Which of {{patterns.response}} is best for a real-time chat application? Explain in 2-3 sentences.",
    prompt_name="recommendation",
)
print(f"[recommendation] tokens={r2.usage.total_tokens} cost=${r2.cost_usd:.6f}")
print(r2.response[:200])
print("...")

[recommendation] tokens=269 cost=$0.000026
For a **real-time chat application**, the **Event-Driven Architecture** is the best choice. This architecture excels in handling asynchronous communication, where events (like new messages) trigger im
...


In [4]:
r3 = ffai.generate_response(
    prompt=(
        "Given the recommendation that {{recommendation.response}}, "
        "list three specific technical challenges a team would face implementing it."
    ),
    prompt_name="challenges",
)
print(f"[challenges] tokens={r3.usage.total_tokens} cost=${r3.cost_usd:.6f}")
print(r3.response[:200])
print("...")

[challenges] tokens=733 cost=$0.000118
Implementing an **Event-Driven Architecture (EDA)** for a real-time chat application presents several technical challenges. Here are three specific ones a team would face:

### 1. **Event Ordering and
...


In [5]:
r4 = ffai.generate_response(
    prompt=(
        "For each challenge in {{challenges.response}}, suggest one concrete mitigation "
        "strategy. Be brief -- one sentence each."
    ),
    prompt_name="mitigations",
)
print(f"[mitigations] tokens={r4.usage.total_tokens} cost=${r4.cost_usd:.6f}")
print(r4.response[:200])
print("...")

[mitigations] tokens=683 cost=$0.000045
For **Event Ordering and Consistency**, implement a **partitioned event log with sequence numbers per chat room** to enforce strict ordering and detect gaps.
...


In [6]:
r5 = ffai.generate_response(
    prompt="Summarize the entire discussion in exactly three bullet points.",
    prompt_name="summary",
    history=["patterns", "recommendation", "challenges", "mitigations"],
)
print(f"[summary] tokens={r5.usage.total_tokens} cost=${r5.cost_usd:.6f}")
print(r5.response)

[summary] tokens=2060 cost=$0.000142
- **Patterns Overview**: The discussion started by listing four key software architecture patterns—Layered Architecture, Microservices Architecture, Event-Driven Architecture (EDA), and Model-View-Controller (MVC)—with concise descriptions of each.

- **Recommendation for Real-Time Chat**: EDA was recommended for real-time chat applications due to its strength in handling asynchronous communication, scalability, and low latency, enabling immediate responses to events like new messages.

- **Challenges and Mitigations**: Three technical challenges of implementing EDA for chat apps were highlighted—event ordering/consistency, scalability/backpressure, and fault tolerance/event loss—along with a mitigation strategy (e.g., partitioned event logs with sequence numbers for ordering).


In [7]:
results = [r1, r2, r3, r4, r5]
total_tokens = sum(r.usage.total_tokens for r in results)
total_cost = sum(r.cost_usd for r in results)
print(f"Pipeline complete: {len(results)} prompts")
print(f"Total tokens: {total_tokens}")
print(f"Total cost:   ${total_cost:.6f}")

Pipeline complete: 5 prompts
Total tokens: 3929
Total cost:   $0.000359


<div class="page-break"></div>

---

## Step 2: Raw History

The raw `history` list records every interaction dict as-is -- including the
full `resolved_prompt` (after interpolation), `usage` object, and `status`.

This is the most detailed view.

In [8]:
raw_df = ffai.history_to_dataframe()
print(f"Raw history: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
print(f"Columns: {raw_df.columns}")
print()
display_cols = [c for c in ["prompt_name", "model", "status", "timestamp", "datetime"] if c in raw_df.columns]
print(raw_df.select(display_cols))

Raw history: 5 rows x 10 columns
Columns: ['prompt', 'response', 'prompt_name', 'timestamp', 'model', 'history', 'status', 'resolved_prompt', 'usage', 'datetime']

shape: (5, 5)
┌────────────────┬──────────────────────┬─────────┬───────────┬────────────────────────────┐
│ prompt_name    ┆ model                ┆ status  ┆ timestamp ┆ datetime                   │
│ ---            ┆ ---                  ┆ ---     ┆ ---       ┆ ---                        │
│ str            ┆ str                  ┆ str     ┆ f64       ┆ datetime[μs]               │
╞════════════════╪══════════════════════╪═════════╪═══════════╪════════════════════════════╡
│ patterns       ┆ mistral-small-latest ┆ success ┆ 1.7801e9  ┆ 2026-05-29 18:11:00.730579 │
│ recommendation ┆ mistral-small-latest ┆ success ┆ 1.7801e9  ┆ 2026-05-29 18:11:01.754040 │
│ challenges     ┆ mistral-small-latest ┆ success ┆ 1.7801e9  ┆ 2026-05-29 18:11:06.265481 │
│ mitigations    ┆ mistral-small-latest ┆ success ┆ 1.7801e9  ┆ 2026-05-29 18:

In [9]:
first = raw_df.row(0, named=True)
print(f"First interaction prompt_name: {first.get('prompt_name')}")
print(f"First interaction model:       {first.get('model')}")
print(f"First interaction status:       {first.get('status')}")
print(f"First interaction timestamp:    {first.get('timestamp')}")
resolved = first.get("resolved_prompt", "")
print(f"Resolved prompt (first 120 chars): {resolved[:120]}")

First interaction prompt_name: patterns
First interaction model:       mistral-small-latest
First interaction status:       success
First interaction timestamp:    1780078260.7305794
Resolved prompt (first 120 chars): List exactly four software architecture patterns with a one-sentence description of each.


<div class="page-break"></div>

---

## Step 3: Ordered History

The ordered history provides a sequenced view with **cleaned** prompts and
responses (RAG sections stripped, whitespace normalized). It also includes
the `history` dependency list for each interaction.

In [10]:
ordered_df = ffai.ordered_history_to_dataframe()
print(f"Ordered history: {ordered_df.shape[0]} rows x {ordered_df.shape[1]} columns")
print(f"Columns: {ordered_df.columns}")
print()
display_cols = [c for c in ["sequence_number", "prompt_name", "model", "datetime"] if c in ordered_df.columns]
print(ordered_df.select(display_cols))

Ordered history: 5 rows x 8 columns
Columns: ['sequence_number', 'model', 'timestamp', 'prompt_name', 'prompt', 'response', 'history', 'datetime']

shape: (5, 4)
┌─────────────────┬────────────────┬──────────────────────┬────────────────────────────┐
│ sequence_number ┆ prompt_name    ┆ model                ┆ datetime                   │
│ ---             ┆ ---            ┆ ---                  ┆ ---                        │
│ i64             ┆ str            ┆ str                  ┆ datetime[μs]               │
╞═════════════════╪════════════════╪══════════════════════╪════════════════════════════╡
│ 1               ┆ patterns       ┆ mistral-small-latest ┆ 2026-05-29 18:11:00.731264 │
│ 2               ┆ recommendation ┆ mistral-small-latest ┆ 2026-05-29 18:11:01.754170 │
│ 3               ┆ challenges     ┆ mistral-small-latest ┆ 2026-05-29 18:11:06.265692 │
│ 4               ┆ mitigations    ┆ mistral-small-latest ┆ 2026-05-29 18:11:07.036048 │
│ 5               ┆ summary        ┆ 

In [11]:
summary_row = ordered_df.filter(ordered_df["prompt_name"] == "summary")
if summary_row.shape[0] > 0:
    row = summary_row.row(0, named=True)
    print(f"Summary interaction dependencies (history): {row.get('history')}")
    print(f"Summary cleaned prompt (first 100 chars): {str(row.get('prompt', ''))[:100]}")
else:
    print("No summary row found")

Summary interaction dependencies (history): ['patterns', 'recommendation', 'challenges', 'mitigations']
Summary cleaned prompt (first 100 chars): Summarize the entire discussion in exactly three bullet points.


<div class="page-break"></div>

---

## Step 4: Prompt-Attribute History

The prompt-attribute history powers `{{name.response}}` interpolation. When a
response is a dict, FFAI explodes it into separate entries -- one per key.

For string responses (our case), it stores one entry per interaction, indexed
by `prompt_name`.

In [12]:
attr_df = ffai.prompt_attr_history_to_dataframe()
print(f"Prompt-attribute history: {attr_df.shape[0]} rows x {attr_df.shape[1]} columns")
print()
display_cols = [c for c in ["prompt_name", "model", "timestamp"] if c in attr_df.columns]
print(attr_df.select(display_cols))

Prompt-attribute history: 5 rows x 7 columns

shape: (5, 3)
┌────────────────┬──────────────────────┬───────────┐
│ prompt_name    ┆ model                ┆ timestamp │
│ ---            ┆ ---                  ┆ ---       │
│ str            ┆ str                  ┆ f64       │
╞════════════════╪══════════════════════╪═══════════╡
│ patterns       ┆ mistral-small-latest ┆ 1.7801e9  │
│ recommendation ┆ mistral-small-latest ┆ 1.7801e9  │
│ challenges     ┆ mistral-small-latest ┆ 1.7801e9  │
│ mitigations    ┆ mistral-small-latest ┆ 1.7801e9  │
│ summary        ┆ mistral-small-latest ┆ 1.7801e9  │
└────────────────┴──────────────────────┴───────────┘


<div class="page-break"></div>

---

## Step 5: Basic Statistics

FFAI provides several built-in statistics methods that return Polars DataFrames.

In [13]:
print("=== Model Usage Stats ===")
print(ffai.get_model_stats_df())
print()
print("=== Prompt Name Usage Stats ===")
print(ffai.get_prompt_name_stats_df())

=== Model Usage Stats ===
shape: (1, 2)
┌──────────────────────┬───────┐
│ model                ┆ count │
│ ---                  ┆ ---   │
│ str                  ┆ i64   │
╞══════════════════════╪═══════╡
│ mistral-small-latest ┆ 5     │
└──────────────────────┴───────┘

=== Prompt Name Usage Stats ===
shape: (5, 2)
┌────────────────┬───────┐
│ prompt_name    ┆ count │
│ ---            ┆ ---   │
│ str            ┆ i64   │
╞════════════════╪═══════╡
│ patterns       ┆ 1     │
│ recommendation ┆ 1     │
│ challenges     ┆ 1     │
│ mitigations    ┆ 1     │
│ summary        ┆ 1     │
└────────────────┴───────┘


In [14]:
print("=== Response Length Stats (by prompt name) ===")
print(ffai.get_response_length_stats())

=== Response Length Stats (by prompt name) ===
shape: (5, 5)
┌────────────────┬─────────────┬────────────┬────────────┬───────┐
│ prompt_name    ┆ mean_length ┆ min_length ┆ max_length ┆ count │
│ ---            ┆ ---         ┆ ---        ┆ ---        ┆ ---   │
│ str            ┆ f64         ┆ u32        ┆ u32        ┆ u64   │
╞════════════════╪═════════════╪════════════╪════════════╪═══════╡
│ challenges     ┆ 2882.0      ┆ 2882       ┆ 2882       ┆ 1     │
│ summary        ┆ 788.0       ┆ 788        ┆ 788        ┆ 1     │
│ patterns       ┆ 749.0       ┆ 749        ┆ 749        ┆ 1     │
│ recommendation ┆ 441.0       ┆ 441        ┆ 441        ┆ 1     │
│ mitigations    ┆ 157.0       ┆ 157        ┆ 157        ┆ 1     │
└────────────────┴─────────────┴────────────┴────────────┴───────┘


In [15]:
print("=== Interaction Counts by Date ===")
print(ffai.interaction_counts_by_date())

=== Interaction Counts by Date ===
shape: (1, 2)
┌────────────┬─────┐
│ date       ┆ len │
│ ---        ┆ --- │
│ date       ┆ u64 │
╞════════════╪═════╡
│ 1970-01-01 ┆ 5   │
└────────────┴─────┘


<div class="page-break"></div>

---

## Step 6: Token Usage & Cost Aggregation

Each `ResponseResult` carries `usage` (a `TokenUsage` dataclass with
`input_tokens`, `output_tokens`, `total_tokens`) and `cost_usd`.

The `TokenUsage` class supports `+` for aggregation.

In [16]:
import polars as pl

per_turn = pl.DataFrame({
    "prompt_name": ["patterns", "recommendation", "challenges", "mitigations", "summary"],
    "input_tokens": [r1.usage.input_tokens, r2.usage.input_tokens, r3.usage.input_tokens, r4.usage.input_tokens, r5.usage.input_tokens],
    "output_tokens": [r1.usage.output_tokens, r2.usage.output_tokens, r3.usage.output_tokens, r4.usage.output_tokens, r5.usage.output_tokens],
    "total_tokens": [r1.usage.total_tokens, r2.usage.total_tokens, r3.usage.total_tokens, r4.usage.total_tokens, r5.usage.total_tokens],
    "cost_usd": [r1.cost_usd, r2.cost_usd, r3.cost_usd, r4.cost_usd, r5.cost_usd],
    "duration_ms": [r1.duration_ms, r2.duration_ms, r3.duration_ms, r4.duration_ms, r5.duration_ms],
})

totals = {
    "prompt_name": "TOTAL",
    "input_tokens": per_turn["input_tokens"].sum(),
    "output_tokens": per_turn["output_tokens"].sum(),
    "total_tokens": per_turn["total_tokens"].sum(),
    "cost_usd": round(per_turn["cost_usd"].sum(), 6),
    "duration_ms": round(per_turn["duration_ms"].sum(), 1),
}

per_turn = pl.concat([per_turn, pl.DataFrame([totals])], how="diagonal")
print(per_turn)

shape: (6, 6)
┌────────────────┬──────────────┬───────────────┬──────────────┬──────────┬─────────────┐
│ prompt_name    ┆ input_tokens ┆ output_tokens ┆ total_tokens ┆ cost_usd ┆ duration_ms │
│ ---            ┆ ---          ┆ ---           ┆ ---          ┆ ---      ┆ ---         │
│ str            ┆ i64          ┆ i64           ┆ i64          ┆ f64      ┆ f64         │
╞════════════════╪══════════════╪═══════════════╪══════════════╪══════════╪═════════════╡
│ patterns       ┆ 39           ┆ 145           ┆ 184          ┆ 0.000028 ┆ 1508.9      │
│ recommendation ┆ 187          ┆ 82            ┆ 269          ┆ 0.000026 ┆ 1011.0      │
│ challenges     ┆ 120          ┆ 613           ┆ 733          ┆ 0.000118 ┆ 4498.9      │
│ mitigations    ┆ 651          ┆ 32            ┆ 683          ┆ 0.000045 ┆ 754.9       │
│ summary        ┆ 1906         ┆ 154           ┆ 2060         ┆ 0.000142 ┆ 1320.2      │
│ TOTAL          ┆ 2903         ┆ 1026          ┆ 3929         ┆ 0.000359 ┆ 9093.9    

In [17]:
total_usage = r1.usage + r2.usage + r3.usage + r4.usage + r5.usage
print(f"Aggregated via TokenUsage.__add__: {total_usage.input_tokens} in / {total_usage.output_tokens} out / {total_usage.total_tokens} total")
print(f"Same as sum of individual totals:   {sum(r.usage.total_tokens for r in results)} total")

Aggregated via TokenUsage.__add__: 2903 in / 1026 out / 3929 total
Same as sum of individual totals:   3929 total


<div class="page-break"></div>

---

## Summary

| History Store | Access Method | Key Feature |
|---|---|---|
| Raw history | `ffai.history` / `history_to_dataframe()` | Full detail: resolved prompt, usage, status |
| Clean history | `ffai.clean_history` / `clean_history_to_dataframe()` | Same structure as raw (cleaned by recorder) |
| Ordered history | `ffai.ordered_history` / `ordered_history_to_dataframe()` | Sequence numbers, cleaned text, dependency lists |
| Prompt-attribute history | `ffai.prompt_attr_history` / `prompt_attr_history_to_dataframe()` | Indexed by name, dict-explosion for structured responses |

| Statistic | Method | Returns |
|---|---|---|
| Model usage | `get_model_stats_df()` | `{model, count}` DataFrame |
| Prompt name usage | `get_prompt_name_stats_df()` | `{prompt_name, count}` DataFrame |
| Response lengths | `get_response_length_stats()` | `{prompt_name, mean/min/max_length, count}` DataFrame |
| Daily counts | `interaction_counts_by_date()` | `{date, len}` DataFrame |
| Token/cost | `r.usage` + `r.cost_usd` per `ResponseResult` | `TokenUsage` is additive via `+` |